# Chapter 4 — LLM and Memory / Chat History

## LLM vs Human Memory
When talking to another human being, conversation flows naturally.
Imagine you ask someone whether they can help you with your math homework.
You might first ask, “Do you have time tomorrow?” The other person replies, “Yes.”
After that, you can simply ask, “Can you help me with my math homework?”

You do **not** need to restate the entire previous conversation, such as:

> “Earlier I asked whether you have time tomorrow. You said yes.
> Therefore, I will now ask whether you can help me with my math homework.”

Humans naturally remember the context of a conversation.
Large Language Models (LLMs), however, do not.

---

Another interesting aspect of human cognition is **long-term memory**.
For example, you might have visited Disneyland as a child. You do not constantly think about that memory. However, if someone asks, “Have you ever been to Disneyland?”, you can immediately recall the experience.

Humans store enormous amounts of information in long-term memory and retrieve it **only when relevant**.

Unfortunately, LLMs do not work this way.

LLMs operate with a **context window**, which you can think of as the maximum number of tokens (roughly words or word pieces) the model can read in a single request. In order for an LLM to “remember” something, the information must appear inside this context window.

This means that the conversation history usually needs to be sent to the model again with each new request. If the conversation becomes too long and exceeds the context window limit, the model can no longer process the entire history.

In other words, **LLMs do not have memory by themselves**.

---

This limitation has important implications when building LLM applications.
If we want an LLM system to remember past interactions, user preferences, or previous knowledge, we must build an **external memory system**.

However, simply storing everything is not enough. Because the context window is limited, we also need a mechanism to **retrieve only the relevant information** for the current query.

For example, if the current discussion is about Disneyland, the system should retrieve all relevant memories related to Disneyland from storage and include them in the prompt sent to the LLM.

This process—**storing information externally and retrieving relevant pieces when needed**—is the foundation of memory systems used in modern LLM applications.

## Memory vs. Knowledge from Training

One thing that often confuses people is that LLMs sometimes **appear to have memory**, even when they do not.

For example, if you ask an LLM:

> "What is the capital city of the UK?"

The model will answer:

> "London."

It may look as if the model is **remembering a fact**. However, this is not a real memory in the sense humans use the term.

During pretraining, the model is exposed to massive amounts of text data and learns to predict the next token in a sequence. In other words, the model learns **statistical patterns of language**. It does not store facts in a database or retrieve them from a memory store.

Instead, when the model sees the question:

> "What is the capital city of the UK?"

it computes the probability of all possible next tokens. Among all possible English words, the token **"London"** has the highest probability given this context, because that pattern frequently appeared in the training data.


## 4.1 The Simplest Memory: Chat History as a List

The easiest way to give an LLM "memory" is simply to **store the conversation history and send it back to the model with each new request**.

This approach does not require any special framework or database. We just keep all previous messages in a Python list and include them in the next model call.

Let's start by initializing a model. In this example we connect to a local LM Studio server.



In [33]:
import os
from langchain.chat_models import init_chat_model

base_url = os.getenv("LMSTUDIO_BASE_URL", "http://localhost:1234/v1")
api_key = os.getenv("LMSTUDIO_API_KEY", "lm-studio")

model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="openai",
    base_url=base_url,
    api_key=api_key,
    output_version="responses/v1"
)

In [34]:
response1 = model.invoke("Is Boston in the US?")
print(response1.text)

print("="*20,'now ask a follow up questions:',"="*20)
response2 = model.invoke("Sorry I missed your reply, can you tell me again?")
print(response2.text)

Yes, Boston is a city in the United States. It’s the capital of Massachusetts and one of the country’s oldest cities, located on the eastern coast in New England.
==================== now ask a follow up questions: ====================
I’m happy to help! What would you like me to explain again?


> In the example above, LLM does not 'remember' the conversation.

> Now let's store the conversation history manually using a Python list.

In [39]:
history = []

user_msg = "Is Boston in the US?"
print("User asks:", user_msg)
# Add user question to history
history.append({"role": "user", "content": user_msg})
response = model.invoke(history)
print("\nLLM response:")
print(response.text)
# Save assistant response into history
history.append({"role": "assistant", "content": response.text})
# Second user message
user_msg = "Sorry I missed your reply, can you tell me again?"
print("\nUser asks:", user_msg)
# Add new user message
history.append({"role": "user", "content": user_msg})
# Send full history to model
response = model.invoke(history)
print("\nLLM response:")
print(response.text)
print("What the user normally sees in the chat is printed above")

print("\n" + "="*100)
print("\n**But what we actually send to the LLM is the FULL history"
      "(use last user question as an example):\n")
print(history)

print("="*100)





User asks: Is Boston in the US?

LLM response:
Yes, Boston is a city in the United States. It’s the capital of Massachusetts and one of the country’s oldest and most historic cities.

User asks: Sorry I missed your reply, can you tell me again?

LLM response:
Yes—Boston is a city in the United States, located in the state of Massachusetts.
What the user normally sees in the chat is printed above


**But what we actually send to the LLM is the FULL history(use last user question as an example):

[{'role': 'user', 'content': 'Is Boston in the US?'}, {'role': 'assistant', 'content': 'Yes, Boston is a city in the United States. It’s the capital of Massachusetts and one of the country’s oldest and most historic cities.'}, {'role': 'user', 'content': 'Sorry I missed your reply, can you tell me again?'}]


One important detail is that the **entire conversation history is not normally visible to the user**.

In a typical chat interface, the user only sees the conversation itself:

User: Is Boston in the US?
Assistant: Yes, Boston is a city in the United States.
User: Sorry I missed your reply, can you tell me again?

However, under the hood, the application sends the **entire conversation history** to the model every time a new request is made. Because the model receives the previous messages again, it can understand the context and generate an appropriate reply.

This creates the impression that the model "remembers" previous messages, even though it is simply being given the full conversation again with each request.

## 4.2 Selecting Relevant Memory from History

In the previous section we simply sent the **entire conversation history** to the LLM.
However, as the history grows longer, sending everything becomes inefficient and may exceed the model's **context window**.

A common strategy is to **store all messages but only retrieve the relevant ones** for the current query.

To illustrate this idea, we will implement a very simple memory selection mechanism using a Python list.

### Step 1 — Build the conversation history

Suppose the user first talks about a **movie**, and later introduces a **gene** that happens to have the same name.



In [40]:

history = []

history.append({"role": "user", "content": "Do you know a movie called Nezha?"})
history.append({"role": "assistant", "content": "Yes, Nezha is a popular Chinese animated movie."})

history.append({
    "role": "user",
    "content": "In the movie, Nezha is a powerful character who can perform many magical abilities."
})

history.append({
    "role": "assistant",
    "content": "Yes, the character Nezha is based on Chinese mythology."
})

history.append({
    "role": "user",
    "content": "Interestingly, there is also a gene called Nezha that is important for regulating cell functions."
})

history.append({
    "role": "assistant",
    "content": "That is interesting. Some genes share names with mythological figures."
})

Step 2 — Create a simple retrieval function

We now write a simple function that selects messages related to a specific keyword.

In [41]:
def retrieve_memory(history, keyword):
    selected = []

    for msg in history:
        if keyword.lower() in msg["content"].lower():
            selected.append(msg)

    return selected

Step 3 — Retrieve only gene-related memory

Suppose the user now asks:

"What is the Nezha gene?"

Instead of sending the entire history, we retrieve only the messages related to either gene or mythology.

In [45]:
relevant_memory_mythology = retrieve_memory(history, "mythology")
print("Retrieved memory related to mythology:")
for m in relevant_memory_mythology:
    print(m)
print("="*100)
print()
print("let's get all memory related to gene:")
relevant_memory_gene= retrieve_memory(history, "gene")
for m in relevant_memory_gene:
    print(m)

Retrieved memory related to mythology:
{'role': 'assistant', 'content': 'Yes, the character Nezha is based on Chinese mythology.'}

let's get all memory related to gene:
{'role': 'user', 'content': 'Interestingly, there is also a gene called Nezha that is important for regulating cell functions.'}
{'role': 'assistant', 'content': 'That is interesting. Some genes share names with mythological figures.'}


Step 4 — Send the selected memory to the LLM

Now we combine the retrieved memory entries with the new question

In [50]:
query = "What is the Nezha?"
messages = relevant_memory_gene + [
    {"role": "user", "content": query}
]

# Our initial query did not specify which Nezha we are asking about
print('Our initial query did not specify which Nezha we are asking about:', query)
print('Now provide a discussion about the gene, the final query is:')
print(messages)

response = model.invoke(messages)
print('='*20,'LLM reply','='*20)

print(response.text)

Our initial query did not specify which Nezha we are asking about: What is the Nezha?
Now provide a discussion about the gene, the final query is:
[{'role': 'user', 'content': 'Interestingly, there is also a gene called Nezha that is important for regulating cell functions.'}, {'role': 'assistant', 'content': 'That is interesting. Some genes share names with mythological figures.'}, {'role': 'user', 'content': 'What is the Nezha?'}]
==================== LLM reply ====================
The LLM reply is:
**Nezha (also known as *NEZHA*) – a human gene**

| Feature | Details |
|---------|---------|
| **Official symbol** | NEZHA (sometimes written *NEZH*) |
| **Chromosomal location** | Chromosome 6p21.3 (human genome assembly GRCh38) |
| **Gene type** | Protein‑coding gene |
| **Length** | ~3.4 kb of genomic DNA; 2 exons encoding a protein of about 350 amino acids |
| **Protein** | Nezha homolog, predicted to be a small regulatory protein with an N‑terminal signal peptide and a C‑terminal do

In [51]:
print('Now provide a discussion about mythology.The final query is:')
messages = relevant_memory_mythology + [
    {"role": "user", "content": query}
]
print(messages)
print('='*20,'LLM reply','='*20)
response = model.invoke(messages)
print(response.text)

Now provide a discussion about mythology.The final query is:
[{'role': 'assistant', 'content': 'Yes, the character Nezha is based on Chinese mythology.'}, {'role': 'user', 'content': 'What is the Nezha?'}]
==================== LLM reply ====================
**Nezha (哪吒)** is a legendary figure from Chinese folklore, best known as a youthful warrior deity who appears in several classic literary works and has become an enduring icon in Chinese popular culture.  Below is a concise overview of who Nezha is, his mythological background, and why he remains so popular today.

---

## 1. Origins & Mythological Roots

| Aspect | Details |
|--------|---------|
| **First Appearance** | *Fengshen Yanyi* (The Investiture of the Gods, 16th‑century Ming novel) – a retelling of earlier folk tales. |
| **Family** | - *Father*: Li Jing (李靖), a general of the Shang dynasty. <br>- *Mother*: Lady Sun (孫氏). |
| **Birth** | According to legend, Nezha was born from a glowing ball that emerged from his mother’

## 4.3 From Keyword Search to Semantic Search

The previous example showed how to retrieve relevant information from memory.
However, that approach is still too simple, and in practice, it is often not sufficient.

The retrieval function we used was **keyword-based**. This creates several problems.

First, the exact word form may change. For example:

- `gene`
- `genes`
- `Gene`

These words are closely related in meaning, but a simple keyword-matching function may treat them differently.

Second, the user's question may not contain the exact keyword we expect.
For example, the memory may mention **cell regulation**, while the user asks about a **biological factor controlling cell behavior**. Even though the meanings are related, exact keyword matching may fail.

So in real LLM systems, we usually need a better retrieval method.

One common solution is **semantic search**.

Instead of comparing raw words, semantic search attempts to compare the **meaning** of the text. To do this, we first convert each piece of text into a numeric representation called an **embedding vector**.

In simple terms:

- each memory entry is converted into a vector
- the user's query is also converted into a vector
- we then compare vectors to find the memory entries that are **closest in meaning** to the query

This is very different from keyword matching.
With semantic search, two texts do not need to share the exact same words. As long as they express similar ideas, their vectors may be close to each other.

For example, these two sentences may be semantically similar even though the wording is different:

- "Nezha is a gene involved in regulating cell functions."
- "Nezha plays an important role in cell regulation."

A keyword-based retriever might miss this relationship, but a semantic retriever can often identify it.

This is why semantic search is widely used in modern memory systems and Retrieval-Augmented Generation (RAG) pipelines.

In the next section, we will see how to:

1. convert text into vectors,
2. store those vectors,
3. compare them with a query vector,
4. retrieve the most relevant memory entries.

## 4.4 What Is an Embedding?

To perform semantic search, we first need a way to represent text in a numerical form that captures its meaning. This representation is called an **embedding**.

An embedding converts a piece of text into a **vector of numbers**.


Each number in the vector does not have an obvious human interpretation. However, together they represent the **semantic meaning** of the text.

The key idea behind embeddings is that **texts with similar meanings will have similar vectors**. In other words, their vectors will be close to each other in the vector space.

For example, the following two sentences may produce very similar embeddings:

- "Nezha is a gene involved in regulating cell functions."
- "The Nezha gene plays an important role in cell regulation."

Even though the wording is different, their meanings are similar, so their vectors will be close together.

On the other hand, a sentence about the movie character Nezha would likely produce a vector that is **far away** from the gene-related sentences.

Once we convert both the stored memory and the user query into embeddings, we can compute a **similarity score** between vectors. The most similar vectors correspond to the most relevant pieces of memory.

---

### We will use Sentence-BERT to convert text into embeddings

Sentence-BERT documentation:
https://www.sbert.net/

In practice, the workflow for semantic retrieval looks like this:

In [54]:
from sentence_transformers import SentenceTransformer
history = [
    "Alice: I just finished reading a book about machine learning.",
    "Bob: That sounds interesting! I went hiking yesterday.",
    "Jane: I am planning to go to the beach this weekend.",
    "Tom: I will stay home and watch movies.",
]

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(history)
print(embeddings.shape)
print('you can take a quick look at some elements of the embeddings:')
print(embeddings[0][:30])

(4, 384)
you can take a quick look at some elements of the embeddings:
[-0.00305474 -0.017508    0.00330059  0.05280416  0.0270355  -0.00520502
  0.03099986 -0.05880865 -0.0226912   0.01219444 -0.0527579   0.07345674
  0.05302426 -0.07462968 -0.04141366  0.05851284 -0.05061217 -0.02643851
 -0.04223349 -0.09328047 -0.04657337  0.01033305 -0.00894921 -0.00191697
  0.00835001  0.0478383   0.00885494 -0.01157455 -0.05424875 -0.04031479]


In [56]:
from sentence_transformers.util import cos_sim
import numpy as np

query = "What is Jane planning to do this weekend?"
query_embedding = embed_model.encode(query)
similarities = cos_sim(query_embedding, embeddings)

print(similarities)
best_match_index = np.argmax(similarities)
retrieved_message = history[best_match_index]
print("Retrieved memory:", retrieved_message)

tensor([[0.1898, 0.1667, 0.7062, 0.3451]])
Retrieved memory: Jane: I am planning to go to the beach this weekend.


## 4.6 Summary

In this chapter, we used a simple **Python list** to demonstrate the core idea of memory (or chat history) in LLM applications. Each message in the conversation is appended to the list and then sent back to the model when generating the next response. Although this approach is simple, it helps illustrate an important point: LLMs do not truly remember previous interactions; instead, the application **replays relevant information** back to the model.

In practice, most agent frameworks provide utilities to manage memory more conveniently. For example, in **LangGraph**, there are several mechanisms for sharing and persisting information across interactions, including:

- **Store** — for long-term structured storage
- **Checkpoint** — for saving and restoring the state of a conversation or workflow
- **Context** — for passing shared information during execution

**-> We will discuss these mechanisms in more detail in later chapters.**

Memory systems can also be extended beyond simple chat history. For example:

- **Persistent storage** using external databases
- **Vector databases** for semantic retrieval
- **Summarized memory** to compress long conversations
- **Knowledge graphs** to represent structured relationships
- .......

Researchers are also actively exploring more advanced memory architectures. For example, the MEMO paper proposes a more sophisticated memory system for LLM agents.

**Reference:**  
Mem0: Building Production-Ready AI Agents with Scalable Long-Term Memory  
https://arxiv.org/abs/2504.19413

There is no single memory strategy that works best for all applications. The most important task is to design a **memory architecture that fits your specific use case**. Some systems prioritize simplicity and speed, while others require long-term knowledge storage or complex retrieval mechanisms.

Understanding the trade-offs between different memory approaches is an important part of designing effective LLM applications.